### Problem 17 : The enterprise playbook - from a deletion request to a signed certificate

⏱️ ~45 min · 🧭 Stage 7 (Engineering), the top of the ladder · 📄 the book's RAI Scorecard chapter (`resources/book/`), `resources/papers/04-evaluation-and-privacy/2021-thudi-auditable-definitions.pdf`

> 🔗 **Building on Problems 1-16 :** we have exact methods for summation models, Newton steps for linear models,
> SISA, adapters and checkpoints for neural networks, five approximate methods, an audit protocol, a fairness monitor,
> poisoning defences and an LLM attack battery. Each one answered *one* question.

A company does not receive "an unlearning problem". It receives a **request** : an email to the DPO, a ticket from
legal, a contract termination, a security incident. Someone has to decide, before a legal deadline, which models are
affected, which method to use, how to prove it worked, and who signs it off. This problem turns the course into that
process, as code you can adapt.

![Request lifecycle](assets/diagrams/11_request_lifecycle.png)

> 🟢 **In plain words :** a playbook is a recipe for the organisation, not for the model. Who does what, in what order,
> with what evidence, by when. Most unlearning failures in practice are not maths failures. Nobody knew which models
> had seen the data.

> 🔴 **For practitioners :** the six parts : (1) a **lineage registry** of every artefact and the data that reached it,
> (2) a **decision procedure** that picks the cheapest method with an adequate guarantee, (3) a **capacity plan**
> against legal deadlines, (4) an **evidence record / certificate** that a third party can re-verify, (5) **owners**
> for every threshold, and (6) an **Unlearning pillar** in the RAI Scorecard so readiness is measured before a request
> arrives.

**TASK :** Build each part, run four realistic requests through it, issue a certificate, and verify it independently.

In [1]:
import hashlib, json, os
from datetime import date, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mul import lab, train, methods, audit, viz
viz.style()

#### Step 1 : Inventory - the lineage registry

Problem 15 ended with a graph question : *which artefacts descend from Southbank's data?* Here is a minimal registry
for a fictional company's model estate. Every artefact records the **data sources** it was trained on directly and
its **parent** artefacts. Lineage is inherited : whatever reached a parent reached its children.

In [2]:
# Step 1: a lineage registry (in production: your model registry / data catalogue, e.g. MLflow, DVC, OpenLineage)
REGISTRY = {
    # artefact               : family, data it was trained on directly, parents, how it can be unlearned
    "support-base-v1":        dict(family="neural (pre-trained)",  data=["public-web"],                    parents=[],                   design="checkpoint"),
    "support-base-v2":        dict(family="neural (pre-trained)",  data=["tenant:northbank", "tenant:southbank"], parents=["support-base-v1"], design="checkpoint"),
    "adapter-northbank":      dict(family="neural (LoRA adapter)", data=["tenant:northbank"],              parents=["support-base-v2"],  design="per-tenant adapter"),
    "adapter-eastbank":       dict(family="neural (LoRA adapter)", data=["tenant:eastbank"],               parents=["support-base-v2"],  design="per-tenant adapter"),
    "staff-assistant":        dict(family="neural LM (fine-tuned)", data=["hr:directory"],                 parents=["support-base-v1"],  design="checkpoint"),
    "credit-scorer":          dict(family="logistic regression",   data=["customers:applications"],        parents=[],                   design="newton"),
    "door-access-faceid":     dict(family="eigenfaces + gallery",  data=["hr:badge-photos"],               parents=[],                   design="summation"),
    "fraud-net":              dict(family="neural (SISA, 5 shards)", data=["customers:transactions"],      parents=[],                   design="sisa"),
    "spam-filter":            dict(family="naive bayes",           data=["users:feedback"],                parents=[],                   design="summation"),
    "churn-net":              dict(family="neural (monolithic)",   data=["customers:transactions"],        parents=[],                   design="none"),
}

def lineage(artefact):
    """All data sources that reached an artefact, directly or through its parents."""
    a = REGISTRY[artefact]
    return set(a["data"]).union(*[lineage(p) for p in a["parents"]]) if a["parents"] else set(a["data"])

def affected(source):
    """Every artefact that has to be dealt with when `source` must be forgotten."""
    return [name for name in REGISTRY if source in lineage(name)]

print(pd.DataFrame({n: dict(family=a["family"], design=a["design"], lineage=", ".join(sorted(lineage(n)))) for n, a in REGISTRY.items()}).T.to_string())

                                     family              design                                                          lineage
support-base-v1        neural (pre-trained)          checkpoint                                                       public-web
support-base-v2        neural (pre-trained)          checkpoint                   public-web, tenant:northbank, tenant:southbank
adapter-northbank     neural (LoRA adapter)  per-tenant adapter                   public-web, tenant:northbank, tenant:southbank
adapter-eastbank      neural (LoRA adapter)  per-tenant adapter  public-web, tenant:eastbank, tenant:northbank, tenant:southbank
staff-assistant      neural LM (fine-tuned)          checkpoint                                         hr:directory, public-web
credit-scorer           logistic regression              newton                                           customers:applications
door-access-faceid     eigenfaces + gallery           summation                                  

#### Step 2 : Show the issue - four requests arrive in the same week

In [3]:
# Step 2: the requests, and what each one touches
REQUESTS = [
    dict(id="REQ-101", kind="erasure",     source="customers:transactions", subject="customer #88231", received=date(2026, 10, 5), law="GDPR Art. 17"),
    dict(id="REQ-102", kind="contract",    source="tenant:southbank",       subject="Southbank (tenant)", received=date(2026, 10, 6), law="DPA clause 14.3"),
    dict(id="REQ-103", kind="erasure",     source="hr:directory",           subject="employee Dana R.", received=date(2026, 10, 7), law="GDPR Art. 17"),
    dict(id="REQ-104", kind="poisoning",   source="users:feedback",         subject="incident SEC-77", received=date(2026, 10, 8), law="security policy"),
]
for r in REQUESTS:
    print(f"{r['id']} ({r['kind']:9s}, {r['subject']}): affects {affected(r['source'])}")

REQ-101 (erasure  , customer #88231): affects ['fraud-net', 'churn-net']
REQ-102 (contract , Southbank (tenant)): affects ['support-base-v2', 'adapter-northbank', 'adapter-eastbank']
REQ-103 (erasure  , employee Dana R.): affects ['staff-assistant']
REQ-104 (poisoning, incident SEC-77): affects ['spam-filter']


REQ-102 is the Problem 15 cascade : Southbank's data reached `support-base-v2`, so every adapter built on it is affected
too, including Northbank's and Eastbank's, even though their own data is fine. Without the registry, nobody would
have looked at `adapter-eastbank`.

#### Step 3 : Untrain - the decision procedure

The ladder of the course, written as a function. For every (artefact, request) pair it returns the cheapest method that
gives an adequate guarantee, the guarantee type, the evidence required and the problem that teaches it.

In [4]:
# Step 3: the decision procedure distilled from Problems 1-16
def decide(artefact, request):
    a = REGISTRY[artefact]
    direct = request["source"] in a["data"]
    if not direct:
        return ("rebuild from the last clean parent, then re-train this artefact on it", "exact (by process)",
                "lineage record + rebuild logs", "P15")
    if request["kind"] == "poisoning":
        return ("remove found rows + identify the trigger + feature-level removal; monitor ASR", "behavioural",
                "attack success rate on held-out triggered inputs", "P14")
    plan = {
        "summation":          ("subtract sufficient statistics / delete templates", "exact (by construction)", "equality with a from-scratch fit", "P1, P13"),
        "newton":             ("Newton removal within the error budget, else retrain", "exact-ish (certified gradient norm)", "retain-gradient norm + fairness check", "P2, P12"),
        "sisa":               ("SISA unlearn (retrain affected shards from checkpoints)", "exact (by construction)", "partition log + shard hashes", "P7"),
        "per-tenant adapter": ("delete the adapter", "exact (by construction)", "registry entry removed + adapter hash", "P15"),
        "checkpoint":         ("re-fine-tune / re-train from the checkpoint before the data entered", "exact (by process)", "manifest + seed + weight hash (re-verifiable)", "P15, P16"),
        "none":               ("approximate (SCRUB+R / NPO) now + full retrain in the next cycle", "approximate, audited", "Problem 11 audit: band, U-LiRA, relearn, fairness", "P9, P11"),
    }[a["design"]]
    if "LM" in a["family"]:
        plan = (plan[0], plan[1], plan[2] + "; LM attack battery (exposure, prefix forcing, relearn)",
                plan[3] if "P16" in plan[3] else plan[3] + ", P16")
    if request["source"].startswith("hr:badge"):
        plan = (plan[0] + " + non-biometric do-not-enrol key", *plan[1:])
    return plan

rows = []
for r in REQUESTS:
    for art in affected(r["source"]):
        how, guarantee, evidence, ref = decide(art, r)
        rows.append(dict(request=r["id"], artefact=art, method=how, guarantee=guarantee, evidence=evidence, learn_in=ref))
plan = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 70):
    print(plan.to_string(index=False))

request          artefact                                                                        method               guarantee                                                                                             evidence learn_in
REQ-101         fraud-net                       SISA unlearn (retrain affected shards from checkpoints) exact (by construction)                                                                         partition log + shard hashes       P7
REQ-101         churn-net              approximate (SCRUB+R / NPO) now + full retrain in the next cycle    approximate, audited                                                    Problem 11 audit: band, U-LiRA, relearn, fairness  P9, P11
REQ-102   support-base-v2           re-fine-tune / re-train from the checkpoint before the data entered      exact (by process)                                                        manifest + seed + weight hash (re-verifiable) P15, P16
REQ-102 adapter-northbank         rebuild from t

Notice how few rows say "approximate". In a well-designed estate, **most deletions are exact by design** : summation
models, SISA, adapters, and checkpoints. Approximate methods are the fallback for legacy models (`churn-net`), and the
decision procedure pairs them with a scheduled exact retrain so the approximation is temporary.

#### Step 3b : Capacity planning - meeting the deadline

GDPR gives one month to respond (Art. 12(3), extendable by two months for complex cases), CCPA 45 days (extendable
once), India's DPDP Rules set their own timelines. A retrain that takes 3 days is fine for one request. For 400 requests
a month, what matters is the **schedule**. Batch the requests and retrain every N days, and the worst-case delay is N
days of waiting plus the retrain itself.

In [5]:
# Step 3b: cost and worst-case latency of each strategy for a legacy model like churn-net
REQUESTS_PER_MONTH, RETRAIN_DAYS, RETRAIN_COST = 400, 3.0, 12_000        # a 3-day, $12k full retrain (illustrative)
DEADLINE_DAYS = 30
strategies = {
    "retrain per request":            dict(retrains=REQUESTS_PER_MONTH, wait=0),
    "batch: retrain weekly":          dict(retrains=30 / 7, wait=7),
    "batch: retrain every 2 weeks":   dict(retrains=30 / 14, wait=14),
    "batch: retrain monthly":         dict(retrains=1, wait=30),
    "SISA, user-aware (P7: ~5%/req)": dict(retrains=REQUESTS_PER_MONTH * 0.05, wait=0),
    "SISA, random partition (~61%)":  dict(retrains=REQUESTS_PER_MONTH * 0.61, wait=0),
}
cap = pd.DataFrame({k: dict(monthly_cost=v["retrains"] * RETRAIN_COST,
                            worst_case_days=v["wait"] + RETRAIN_DAYS * (0.05 if "user-aware" in k else 0.61 if "random" in k else 1))
                    for k, v in strategies.items()}).T
cap["meets_30_days"] = np.where(cap.worst_case_days <= DEADLINE_DAYS, "✅", "❌")
print(cap.to_string(formatters={"monthly_cost": "${:,.0f}".format, "worst_case_days": "{:.1f}".format}))

                               monthly_cost worst_case_days meets_30_days
retrain per request              $4,800,000             3.0             ✅
batch: retrain weekly               $51,429            10.0             ✅
batch: retrain every 2 weeks        $25,714            17.0             ✅
batch: retrain monthly              $12,000            33.0             ❌
SISA, user-aware (P7: ~5%/req)     $240,000             0.2             ✅
SISA, random partition (~61%)    $2,928,000             1.8             ✅


Batching weekly or fortnightly meets the deadline at about 1% of the cost of per-request retraining. **User-aware SISA**
answers within hours instead of days; here it costs more per month than batching, but it gives every request an exact
answer immediately, with no interim approximate step. Monthly batching is cheapest but misses the deadline once the retrain time is added. This is
where an approximate method earns its place : run SCRUB+R or NPO *immediately* (audited, Problem 11) to stop the
exposure, and let the next batch retrain provide the exact guarantee.

#### Step 4 : Show the result - the unlearning certificate

For REQ-101 on the legacy `churn-net`, the plan was an audited approximate step immediately, then an exact rebuild in
the weekly batch. We execute the exact rebuild for real on the course's lab (Carol stands in for customer #88231) and
issue the certificate. (`fraud-net` gets its own certificate from its SISA unlearning log.) It records *what* was done, *how it can be checked*, and *who* accepted the residual risk.

In [6]:
# Step 4: execute an exact, re-verifiable unlearning and write the certificate
L = lab.make_lab("carol")
RECIPE = dict(model="lab.model_fn (MLP 64-256-256-10)", **lab.FIT, seed=2026)
retain_rows = np.sort(L.retain_idx)

def sha(arr_or_list):
    h = hashlib.sha256()
    for a in (arr_or_list if isinstance(arr_or_list, list) else [arr_or_list]):
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()

unlearned = train.fit(lab.model_fn, L.X_train[retain_rows], L.y_train[retain_rows], seed=RECIPE["seed"], **lab.FIT)
audit_record = json.load(open("results/audit_carol.json"))            # produced by Problem 11

r = REQUESTS[0]
certificate = dict(
    certificate_id="UC-2026-0101", request=r["id"], legal_basis=r["law"],
    data_subject=hashlib.sha256(r["subject"].encode()).hexdigest()[:16],          # pseudonymous reference, not the name
    received=str(r["received"]), deadline=str(r["received"] + timedelta(days=30)), completed=str(r["received"] + timedelta(days=10)),
    artefacts=["churn-net"],
    method="weekly batch retrain from scratch on the retain manifest (exact by process); interim SCRUB+R step audited under Problem 11",
    guarantee="exact: the model is a deterministic function of (recipe, manifest), both recorded below",
    evidence=dict(
        recipe=RECIPE,
        retain_manifest_sha256=sha(retain_rows), rows_removed=int(len(L.forget_idx)),
        forget_rows_absent=bool(np.intersect1d(retain_rows, L.forget_idx).size == 0),
        model_weights_sha256=sha(unlearned.params),
        audit_protocol="Problem 11 (results/audit_carol.json)",
        audit_reference_band=audit_record["gold_band"]["forget_acc"],
        audit_of_this_method=audit_record["results"]["retrain"],
    ),
    residual_risks=["black-box audits are evidence, not proof (Thudi et al. 2022); the guarantee here rests on the process",
                    "backups and logs containing the rows are handled by the data-retention procedure, ticket DR-5531"],
    sign_off=dict(ml_owner="(name, date)", dpo="(name, date)"),
)
os.makedirs("results", exist_ok=True)
with open("results/certificate_UC-2026-0101.json", "w") as f:
    json.dump(certificate, f, indent=1, default=str)
print(json.dumps({k: certificate[k] for k in ["certificate_id", "request", "deadline", "completed", "artefacts", "method"]}, indent=1))
print(json.dumps(certificate["evidence"], indent=1, default=str)[:900], "...")

{
 "certificate_id": "UC-2026-0101",
 "request": "REQ-101",
 "deadline": "2026-11-04",
 "completed": "2026-10-15",
 "artefacts": [
  "churn-net"
 ],
 "method": "weekly batch retrain from scratch on the retain manifest (exact by process); interim SCRUB+R step audited under Problem 11"
}
{
 "recipe": {
  "model": "lab.model_fn (MLP 64-256-256-10)",
  "epochs": 60,
  "lr": 0.001,
  "batch_size": 64,
  "seed": 2026
 },
 "retain_manifest_sha256": "1c8ed4aeef2910a2fbdb054c2a438313af19b4c5b72a34c7fc99f5b979a2f99e",
 "rows_removed": 40,
 "forget_rows_absent": true,
 "model_weights_sha256": "86922e40d55db039c28b65ce166975eee3d0c39caabbb9df8fd17c53b756be83",
 "audit_protocol": "Problem 11 (results/audit_carol.json)",
 "audit_reference_band": {
  "mean": 0.6270833333333333,
  "std": 0.04727835715795924
 },
 "audit_of_this_method": {
  "forget_acc": 0.625,
  "test_acc": 0.9796296296,
  "ulira": 0.075,
  "relearn_steps": 500.0,
  "cost_%": 100.0,
  "FORGET": "ok",
  "PRIVACY": "ok",
  "UTILITY": "o

#### Step 5 : Prove it - an independent verifier re-derives the certificate

The certificate claims the deployed model is *exactly* what you get by training on the logged manifest with the logged
recipe. An auditor who receives the manifest (not the deleted data) can check that claim **bit for bit**, with no trust
in us. Because our NumPy training is fully deterministic given the seed, the model is a function of
`(recipe, manifest)`, and a function can be re-computed.

In [7]:
# Step 5: the auditor's side - only the certificate, the manifest and the training code are needed
cert = json.load(open("results/certificate_UC-2026-0101.json"))
manifest = retain_rows                                                     # handed over with the certificate
checks = {
    "manifest matches its hash": sha(manifest) == cert["evidence"]["retain_manifest_sha256"],
    "deleted rows not in manifest": bool(np.intersect1d(manifest, L.forget_idx).size == 0),
}
replayed = train.fit(lab.model_fn, L.X_train[manifest], L.y_train[manifest], seed=cert["evidence"]["recipe"]["seed"], **lab.FIT)
checks["re-trained weights == certified weights (bit for bit)"] = sha(replayed.params) == cert["evidence"]["model_weights_sha256"]

tampered = train.fit(lab.model_fn, L.X_train, L.y_train, seed=cert["evidence"]["recipe"]["seed"], **lab.FIT)   # a model that kept Carol
checks["a model that kept the data would FAIL this check"] = sha(tampered.params) != cert["evidence"]["model_weights_sha256"]
for k, v in checks.items():
    print(f"{'✅' if v else '❌'}  {k}")

✅  manifest matches its hash
✅  deleted rows not in manifest
✅  re-trained weights == certified weights (bit for bit)
✅  a model that kept the data would FAIL this check


This is the strongest form of evidence in the course : not "no test could tell the difference" (Problem 11's black-box
audit), but **"anyone can recompute it"**. It is exactly what Thudi et al. argue unlearning claims need : a verifiable
*process*. Two engineering conditions make it possible, and both are design choices : **deterministic training**
(fixed seeds, deterministic kernels, which on GPUs need explicit flags) and **manifests** (a hashed list of exactly which
rows went into every artefact). SISA, adapters and checkpoints extend the same idea to cheaper rebuilds.

#### Owners for every number

Every threshold in this course is a **policy decision** wearing a technical costume. Each needs an owner.

In [8]:
# Who owns what (a RACI-style summary: R = responsible, A = accountable, C = consulted)
owners = pd.DataFrame([
    ("lineage registry is complete",                "R",  "C", "",  "",  "A"),
    ("method choice (decide())",                    "R",  "C", "",  "",  "A"),
    ("gold-band width k, min half-width (P6, P11)", "R",  "A", "",  "C", ""),
    ("PRIVACY tolerance for U-LiRA (P11)",          "C",  "A", "C", "",  ""),
    ("FAIRNESS tolerance after deletions (P12)",    "C",  "C", "",  "A", ""),
    ("ASR threshold for poisoning (P14)",           "R",  "",  "",  "",  "A"),
    ("interim approximate method allowed? (P11)",   "C",  "A", "C", "C", ""),
    ("do-not-enrol / re-ingestion block (P13)",     "R",  "A", "C", "",  "C"),
    ("certificate sign-off",                        "R",  "A", "C", "",  ""),
], columns=["decision", "ML engineering", "DPO / privacy", "legal", "RAI / fairness", "security / platform"]).set_index("decision")
print(owners.to_string())

                                            ML engineering DPO / privacy legal RAI / fairness security / platform
decision                                                                                                         
lineage registry is complete                             R             C                                        A
method choice (decide())                                 R             C                                        A
gold-band width k, min half-width (P6, P11)              R             A                    C                    
PRIVACY tolerance for U-LiRA (P11)                       C             A     C                                   
FAIRNESS tolerance after deletions (P12)                 C             C                    A                    
ASR threshold for poisoning (P14)                        R                                                      A
interim approximate method allowed? (P11)                C             A     C          

#### The Unlearning pillar for the RAI Scorecard

The book's RAI Scorecard rates every model on four pillars (Fairness, Explainability, Robustness, Data Provenance) and
gates deployment on their weighted harmonic mean (≥ 0.90 production-ready, < 0.70 blocked). Unlearning readiness
belongs next to Data Provenance, measured **before** a request arrives. A proposed rubric :

In [9]:
# Step 5b: an Unlearning Readiness score, and its effect on the book's aggregate RAI score
RUBRIC = {   # criterion: weight
    "lineage: every training source and parent is registered":                 0.20,
    "exact path exists (summation / SISA / adapter / checkpoint / retrain fits deadline)": 0.25,
    "deterministic training + hashed manifests (re-verifiable certificate)":   0.15,
    "audit protocol automated (band, U-LiRA, relearn, fairness) for approximate paths": 0.15,
    "per-group fairness monitored across the deletion log":                    0.10,
    "re-ingestion block (do-not-train / do-not-enrol list)":                   0.10,
    "backups & logs covered by the retention procedure":                       0.05,
}
def readiness(answers):
    return sum(w for (c, w), ok in zip(RUBRIC.items(), answers) if ok)

estate = {"fraud-net (SISA)": [1, 1, 1, 1, 1, 1, 1], "staff-assistant (LM)": [1, 1, 1, 1, 0, 1, 0],
          "churn-net (legacy)": [1, 0, 0, 1, 1, 0, 1], "support-base-v2": [0, 0, 0, 0, 0, 0, 1]}
U = pd.Series({k: readiness(v) for k, v in estate.items()}, name="unlearning_score")

def rai_aggregate(scores, weights):
    """Weighted harmonic mean, as in the book: one weak pillar drags the whole score down."""
    return sum(weights) / sum(w / max(s, 1e-9) for s, w in zip(scores, weights))

book_pillars = [0.92, 1.0, 0.87, 1.0]            # the book's example credit model: fairness, explainability 3/3, robustness, provenance
rows = {}
for name, u in U.items():
    agg = rai_aggregate(book_pillars + [u], [1, 1, 1, 1, 1])
    rows[name] = dict(unlearning=u, rai_4_pillars=rai_aggregate(book_pillars, [1, 1, 1, 1]), rai_5_pillars=agg,
                      gate="production-ready" if agg >= 0.90 else "RAI Council review" if agg >= 0.70 else "BLOCKED")
print(pd.DataFrame(rows).T.to_string(float_format=lambda v: f"{v:.2f}"))

                     unlearning rai_4_pillars rai_5_pillars                gate
fraud-net (SISA)           1.00          0.94          0.95    production-ready
staff-assistant (LM)       0.85          0.94          0.92    production-ready
churn-net (legacy)         0.50          0.94          0.80  RAI Council review
support-base-v2            0.05          0.94          0.21             BLOCKED


The legacy model and the contaminated base drop into review or get blocked. Not because they are inaccurate or
unfair, but because **the company could not honour a deletion request for them** within the law's deadline with a
defensible guarantee. That is the point of putting unlearning on the scorecard : it turns a future legal risk into a
present engineering task.

----
#### OUTCOME : the playbook on one page

| Stage | What to do | Built in |
|---|---|---|
| **Before any request** | register lineage; prefer exact-by-design (summation models, SISA, adapters, checkpoints); deterministic training + manifests; score the Unlearning pillar | P1-3, P7, P15, P17 |
| **Intake** | verify the requester; find every affected artefact through lineage, including descendants | P15, P17 |
| **Decide** | cheapest method with an adequate guarantee; approximate only as an interim step with a scheduled exact rebuild | P6-P11, P17 |
| **Execute** | exact update / SISA / adapter deletion / checkpoint rebuild; approximate + audit if needed | P1-P16 |
| **Verify** | exact: re-verifiable certificate; approximate: band + U-LiRA + relearn + fairness (+ LM attacks) | P5, P6, P11, P12, P16 |
| **Special cases** | biometric: templates + do-not-enrol; poisoning: behaviour + ASR; LLM facts: refusal ≠ forgetting | P13, P14, P16 |
| **Close** | certificate, sign-off, re-ingestion block, retention of backups, fairness monitor update | P12, P17 |

#### Key Takeaways :

- Most unlearning failures are **organisational** : unknown lineage, no clean checkpoint, no owner for the threshold.
- A **lineage registry** turns "what must be unlearned?" into a graph query, and reveals cascades.
- A good estate makes **most deletions exact by design**. Approximate methods are interim measures with scheduled
  exact rebuilds.
- Meet deadlines with **batching and SISA**, not with per-request retraining.
- The best certificate is **re-verifiable** : deterministic training + hashed manifests let anyone recompute the model.
- Put **Unlearning** on the RAI Scorecard, so readiness is measured before the first request arrives.

#### TASK :
1. Add a new artefact `"churn-distilled"`, a small model distilled from `churn-net`'s outputs (no direct training
   data). What does `affected("customers:transactions")` return now? Does distillation launder the data? (It does
   not : the teacher's knowledge, including memorised examples, can transfer.)
2. Change `REQUESTS_PER_MONTH` to 5,000 and `RETRAIN_DAYS` to 10. Which strategies still meet the 30-day deadline? What
   would you change in the architecture?
3. Break determinism : retrain with `seed=None`. Which verification check fails, and what weaker evidence would you
   have to fall back to? (Hint : Problem 11.)
4. Score one real model you work with on the Unlearning rubric. What is the cheapest change that moves it above 0.7?

#### ❓ Where to go from here
You have climbed the whole ladder : from deleting a row in k-NN, to certifying the removal of a person from a
production system. Test yourself with [PRACTICE.md](PRACTICE.md), read the papers in `resources/`, and when you meet
a new model type, ask the first question of this course : **what property of this model would make forgetting a
subtraction, and how close can we get to it?**

Want to go further? **Stage 8** takes the same ideas to new kinds of systems, starting with recommender systems in
**Problem 18**.